# Réplica do pipeline do colega — nossa GPU

Objetivo: isolar a variável **hardware**. O notebook `perplexity_colab.ipynb` roda o *nosso* `run_benchmark.py` (word_perplexity do harness, n=100 reps). Este aqui roda o **mesmo código do colega** (`joaov1tor/tcc-ptq-llms`) na GPU do Colab, exatamente como ele rodou no DGX Spark.

Diferença entre os dois conjuntos de resultados que vamos ter:

| | **Nós (pipeline próprio)** | **Colega (DGX)** | **Nós (esta réplica)** |
| --- | --- | --- | --- |
| Script PPL | `scripts/run_perplexity.py` (harness `wikitext_document_level`, `word_perplexity`) | `scripts/ppl_wikitext2.py` (token-level, `wikitext-2-raw-v1/test`, janelas 2048) | **mesmo que o colega** |
| Script custo | `scripts/run_benchmark.py` (prompts × reps, n=100) | `scripts/bench_cost.py` (prompt 512 do validation, 10 reps × 3 rodadas, n=30) | **mesmo que o colega** |
| Calibração INT4 | C4 shard 00000, 128 docs ≥ 8000 chars, `concat=2048` | **igual** | **igual** |
| GPU | A100/L4 (Colab) | GB10 (DGX Spark) | **A100/L4 (Colab)** |

Depois de rodar, podemos chamar `python scripts/analyze_stats.py --source drive --run-dir <esta pasta>` e comparar com o run do colega lado a lado, controlando que *só o hardware mudou*.

In [ ]:
!nvidia-smi

## 1. Setup

Baixa o repo do colega e instala as dependências. **O repo `joaov1tor/tcc-ptq-llms` é privado**: precisa do secret `GITHUB_TOKEN` (o mesmo PAT com scope `repo` do notebook principal; Notebook access ligado), e a conta dona do token precisa ter acesso ao repo.

In [ ]:
# Baixa o repo (PRIVADO) do colega via tarball da API do GitHub, autenticado.
# `git clone` / curl anônimo dão erro (404 / "could not read Username") em repo privado.
import os, subprocess, tarfile, shutil
from pathlib import Path
from google.colab import userdata

COLEGA_DIR = Path("/content/tcc-ptq-llms")
if not (COLEGA_DIR / "scripts/bench_cost.py").exists():
    try:
        gh_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        gh_token = None
    assert gh_token, ("Crie o secret GITHUB_TOKEN (PAT com scope repo) no Colab "
                      "(ícone de chave) e ligue Notebook access.")
    COLEGA_DIR.parent.mkdir(parents=True, exist_ok=True)
    tarball = Path("/tmp/colega-repo.tar.gz")
    r = subprocess.run(
        ["curl", "-sSL", "-w", "%{http_code}", "-o", str(tarball),
         "-H", f"Authorization: Bearer {gh_token}",
         "-H", "Accept: application/vnd.github+json",
         "https://api.github.com/repos/joaov1tor/tcc-ptq-llms/tarball/main"],
        capture_output=True, text=True,
    )
    assert r.stdout.strip() == "200", (
        f"GitHub respondeu HTTP {r.stdout.strip()}: token sem scope 'repo', expirado, "
        "ou a conta do token não tem acesso a joaov1tor/tcc-ptq-llms.")
    with tarfile.open(tarball) as tf:
        tf.extractall("/tmp/colega-extract")
    extracted = next(Path("/tmp/colega-extract").iterdir())
    if COLEGA_DIR.exists():
        shutil.rmtree(COLEGA_DIR)
    shutil.move(str(extracted), str(COLEGA_DIR))
    tarball.unlink()
    shutil.rmtree("/tmp/colega-extract", ignore_errors=True)

%cd {COLEGA_DIR}

required = ["scripts/bench_cost.py", "scripts/ppl_wikitext2.py",
            "scripts/quantize_int8.py", "scripts/quantize_awq_c2048.py",
            "scripts/quantize_gptq_c2048.py"]
missing = [p for p in required if not Path(p).exists()]
assert not missing, f"scripts do colega não extraíram: {missing}"
print("OK — repo em", COLEGA_DIR, "com", len(required), "scripts.")

In [ ]:
# Deps do pipeline do colega:
#   transformers, datasets, numpy -> já vêm no Colab
#   bitsandbytes -> INT8 LLM.int8()
#   ninja -> gptqmodel compila kernel AWQ/Marlin via JIT (sem ele quebra com "Ninja is required")
#   optimum -> transformers 5.x exige para carregar artefatos GPTQ
#   gptqmodel + libs do stack dele, --no-deps para não quebrar o ABI pré-instalado do Colab
!pip install -q bitsandbytes
!pip install -q ninja optimum
!pip install -q --no-deps gptqmodel device-smi logbar tokenicer defuser pypcre

# Rede de segurança idêntica à do nosso notebook: se o torch do Colab foi atualizado em outra rodada,
# o trio torch/torchvision/torchaudio pode desalinhar e quebrar o import do transformers.
import torch as _t
_cu_major = int((_t.version.cuda or "0.0").split(".")[0])
import importlib.metadata as _md
try:
    _tv_cu = _md.version("torchvision").split("+cu")[-1][:2]
except _md.PackageNotFoundError:
    _tv_cu = ""
if str(_cu_major) not in _tv_cu:
    print(f"torch cu={_t.version.cuda} vs torchvision cu={_tv_cu} — realinhando trio...")
    get_ipython().system("pip install -q --upgrade torch torchvision torchaudio")

print("\n⚠️  Reinicie o runtime agora (Runtime → Restart runtime) e volte para a próxima célula.")

In [ ]:
# Após restart: carrega HF_TOKEN e volta para o diretório do repo do colega.
import os, shutil
from pathlib import Path
from google.colab import userdata

COLEGA_DIR = Path("/content/tcc-ptq-llms")
assert COLEGA_DIR.exists(), f"Rode a célula de setup antes — {COLEGA_DIR} não existe."

# Se uma rodada anterior baixou os modelos para /content/models/ (quando o %cd falhou),
# aproveita e move para dentro do repo do colega para não re-baixar os ~30 GiB.
old_models = Path("/content/models")
new_models = COLEGA_DIR / "models"
if old_models.exists() and not new_models.exists():
    print(f"[fix] movendo {old_models} -> {new_models} (de rodada anterior)")
    shutil.move(str(old_models), str(new_models))

try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN carregado do secret do Colab")
except Exception:
    print("sem HF_TOKEN: Qwen3-8B-Base baixa, Meta-Llama-3-8B não (precisa aprovar o acesso no HF).")

%cd {COLEGA_DIR}
import torch
assert torch.cuda.is_available(), "GPU não detectada — selecione A100 ou L4 no Runtime."
print(torch.cuda.get_device_name(0))

## 2. Baixar modelos FP16

O colega mantém os pesos em `models/<Nome>` no DGX. Replicamos a mesma estrutura: assim os scripts dele rodam sem flags adicionais.

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

TARGETS = [
    ("Qwen/Qwen3-8B-Base", MODELS_DIR / "Qwen3-8B-Base", "qwen3-8b-base"),
    ("meta-llama/Meta-Llama-3-8B", MODELS_DIR / "Meta-Llama-3-8B", "llama3-8b"),
]

for repo_id, local, slug in TARGETS:
    if (local / "config.json").exists():
        print(f"[skip] {repo_id} já em {local}")
        continue
    print(f"[dl]   {repo_id} -> {local}")
    snapshot_download(repo_id=repo_id, local_dir=str(local), local_dir_use_symlinks=False)
print("prontos:", [str(t[1]) for t in TARGETS])

## 3. Quantizar (INT8 + INT4-AWQ + INT4-GPTQ)

Chama direto os scripts do colega. Nome das pastas de saída idêntico ao padrão dele (`<Nome>-int8`, `<Nome>-awq4`, `<Nome>-gptq4`) para o `run_noite.sh` dele rodar sem mexer.

In [ ]:
# INT8 em LLM.int8() do bitsandbytes (quantize_int8.py — 8 linhas, salva pretrained)
for repo_id, local, slug in TARGETS:
    dst = Path(f"models/{local.name}-int8")
    if (dst / "config.json").exists():
        print(f"[skip] {dst}")
        continue
    print(f"[int8] {local} -> {dst}")
    !python scripts/quantize_int8.py {local} {dst}

In [ ]:
# INT4-GPTQ (c2048) — sym=True, g128, desc_act=False, calibração C4 x128 docs seed=0, concat=2048
for repo_id, local, slug in TARGETS:
    dst = Path(f"models/{local.name}-gptq4")
    if (dst / "config.json").exists():
        print(f"[skip] {dst}")
        continue
    print(f"[gptq] {local} -> {dst}")
    !python scripts/quantize_gptq_c2048.py {local} {dst} --nsamples 128 --seed 0

In [ ]:
# INT4-AWQ (c2048) — sym=False (zero-point), g128, mesma calibração
for repo_id, local, slug in TARGETS:
    dst = Path(f"models/{local.name}-awq4")
    if (dst / "config.json").exists():
        print(f"[skip] {dst}")
        continue
    print(f"[awq]  {local} -> {dst}")
    !python scripts/quantize_awq_c2048.py {local} {dst} --nsamples 128 --seed 0

### Opcional: salvar pesos quantizados no Drive

Se a sessão cair, as próximas rodadas pulam a recalibração se os `models/*-{int8,gptq4,awq4}/` existirem localmente. Para proteger contra queda, copie para o Drive agora.

In [ ]:
SAVE_WEIGHTS = False  # ligue se quiser proteção contra queda da sessão

if SAVE_WEIGHTS:
    from google.colab import drive
    import shutil
    drive.mount("/content/drive")
    dst = Path("/content/drive/MyDrive/tcc-ptq-colega-replica/models")
    dst.mkdir(parents=True, exist_ok=True)
    for p in Path("models").glob("*-int8") :
        shutil.copytree(p, dst / p.name, dirs_exist_ok=True)
    for p in Path("models").glob("*-gptq4"):
        shutil.copytree(p, dst / p.name, dirs_exist_ok=True)
    for p in Path("models").glob("*-awq4"):
        shutil.copytree(p, dst / p.name, dirs_exist_ok=True)
    print("pesos quantizados copiados para", dst)

## 4. Perplexidade (WikiText-2 token-level)

Protocolo do colega: `Salesforce/wikitext/wikitext-2-raw-v1/test`, concatenação por `\n\n`, janelas de 2048 tokens, `PPL = exp(NLL / tokens)`. Comparável às tabelas do GPTQ/AWQ paper (Llama 3 8B ≈ 6,1; Qwen3-8B-Base ≈ 6,99).

In [ ]:
# Saída em replica_out/ (NÃO em resultados/: o repo do colega já traz resultados/ com os JSONs do DGX dele,
# e o `if not exists()` abaixo pularia todas as medições). Mesmo esquema de nomes do DGX dele.
Path("replica_out").mkdir(exist_ok=True)

for repo_id, local, slug in TARGETS:
    print(f"\n===== {slug} PPL =====")
    out_fp16 = Path(f"replica_out/{slug}_fp16.json")
    out_i8   = Path(f"replica_out/{slug}_int8.json")
    out_gptq = Path(f"replica_out/{slug}_gptq4.json")
    out_awq  = Path(f"replica_out/{slug}_awq4.json")

    if not out_fp16.exists():
        !python scripts/ppl_wikitext2.py --model {local} --quant none --dtype float16 --out {out_fp16}
    if not out_i8.exists():
        !python scripts/ppl_wikitext2.py --model models/{local.name}-int8 --quant pre --out {out_i8}
    if not out_gptq.exists():
        !python scripts/ppl_wikitext2.py --model models/{local.name}-gptq4 --quant pre --out {out_gptq}
    if not out_awq.exists():
        !python scripts/ppl_wikitext2.py --model models/{local.name}-awq4 --quant pre --out {out_awq}

## 5. Benchmark de custo (TTFT, tokens/s) — 3 rodadas × 10 reps = n=30

Protocolo do colega: prompt de 512 tokens do **split de validation** (não do test, para não vazar a referência da PPL), `max_new_tokens=128`, greedy, warmup=3, reps=10. Rodado **3 vezes** (`cost.json`, `cost_r2.json`, `cost_r3.json`) para o adapter `drive` do nosso `stats_adapter.py` concatenar em n=30.

In [ ]:
# Loop r1/r2/r3 × 2 modelos × 4 precisões — bloco longo, mas é só sequenciar o bench_cost.py do colega.
# Dá para re-rodar sem apagar: o pulado funciona por arquivo-destino existente.
REPS_SUFFIX = ["", "_r2", "_r3"]  # "" vira cost.json; "_r2" vira cost_r2.json; etc.

for repo_id, local, slug in TARGETS:
    print(f"\n===== {slug} COST =====")
    for suf in REPS_SUFFIX:
        # FP16
        out_fp16 = Path(f"replica_out/{slug}_fp16_cost{suf}.json")
        if not out_fp16.exists():
            !python scripts/bench_cost.py --model {local} --quant none --dtype float16 --out {out_fp16}
        else:
            print(f"[skip] {out_fp16}")
        # INT8
        out_i8 = Path(f"replica_out/{slug}_int8_cost{suf}.json")
        if not out_i8.exists():
            !python scripts/bench_cost.py --model models/{local.name}-int8 --quant pre --out {out_i8}
        else:
            print(f"[skip] {out_i8}")
        # INT4-GPTQ
        out_gptq = Path(f"replica_out/{slug}_gptq4_cost{suf}.json")
        if not out_gptq.exists():
            !python scripts/bench_cost.py --model models/{local.name}-gptq4 --quant pre --out {out_gptq}
        else:
            print(f"[skip] {out_gptq}")
        # INT4-AWQ
        out_awq = Path(f"replica_out/{slug}_awq4_cost{suf}.json")
        if not out_awq.exists():
            !python scripts/bench_cost.py --model models/{local.name}-awq4 --quant pre --out {out_awq}
        else:
            print(f"[skip] {out_awq}")

## 6. Copiar resultados para o Drive

Pasta espelho de `results/drive_reference/` do nosso repo: `tcc-ptq-colega-replica-colab/<timestamp>/`. Depois é só rodar `python scripts/analyze_stats.py --source drive --run-dir <essa pasta>` localmente para gerar o `stats.md` da réplica.

In [ ]:
import shutil
from datetime import datetime, timezone
from google.colab import drive

drive.mount("/content/drive")

RUN_DIR = Path(f"/content/drive/MyDrive/tcc-ptq-colega-replica-colab/{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}")
RUN_DIR.mkdir(parents=True, exist_ok=True)
# Trava de segurança: a réplica tem que ter sido medida NESTA GPU, não copiada do DGX do colega.
import json
gpus = {json.load(open(f)).get("gpu") for f in Path("replica_out").glob("*.json")}
assert gpus and not any("GB10" in str(g) for g in gpus), f"replica_out/ tem dados do DGX (GPUs: {gpus})"
print("GPUs nos resultados:", gpus)
shutil.copytree("replica_out", RUN_DIR / "resultados", dirs_exist_ok=True)
print("resultados copiados para", RUN_DIR / "resultados")

In [ ]:
# Baixar como .zip, caso prefira pegar sem passar pelo Drive.
!zip -qr colega_replica_results.zip replica_out
from google.colab import files
files.download("colega_replica_results.zip")

## 7. Rodar a análise estatística (localmente, depois)

No seu Mac, apontando para a pasta que acabou de ser copiada para o Drive:

```bash
RUN="<DRIVE>/tcc-ptq-colega-replica-colab/<timestamp>/resultados"
.venv/bin/python scripts/analyze_stats.py \
    --run-dir "$RUN" \
    --source drive \
    --out-dir results/colega_replica_colab/stats
```

Depois dá para comparar três colunas lado a lado: **nossos (pipeline próprio) × colega (DGX) × réplica (nosso código do colega na nossa GPU)**. Diferença entre a réplica e o DGX = **efeito do hardware**; diferença entre a réplica e os nossos (pipeline próprio) = **efeito do protocolo** (word vs token PPL, n=100 vs n=30, prompts × reps vs 3 arquivos).